# TeleComX — Customer Churn Analysis
## Notebook 2: SQL Analysis
**Analyst:** Adedayo Adeyinka A 
**Date:** June 2026  
**Objective:** Use SQL to answer key business questions about customer churn.

In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned dataset
df = pd.read_csv('../data/telecomx_clean.csv')

# Load into SQLite database (in memory)
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False, if_exists='replace')

print("Database ready. Table 'customers' loaded.")
print(f"Rows: {len(df):,}")

Database ready. Table 'customers' loaded.
Rows: 7,043


In [2]:
# Q1: What is the overall churn rate?
query1 = """
SELECT 
    Churn,
    COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS percentage
FROM customers
GROUP BY Churn
ORDER BY Churn DESC;
"""

q1 = pd.read_sql_query(query1, conn)
print("Q1: Overall Churn Rate")
print("=" * 35)
print(q1.to_string(index=False))

Q1: Overall Churn Rate
Churn  customer_count  percentage
  Yes            1869       26.54
   No            5174       73.46


In [3]:
# Q2: Does contract type affect churn?
query2 = """
SELECT 
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC;
"""

q2 = pd.read_sql_query(query2, conn)
print("Q2: Churn by Contract Type")
print("=" * 50)
print(q2.to_string(index=False))

Q2: Churn by Contract Type
      Contract  total_customers  churned  churn_rate_pct
Month-to-month             3875     1655           42.71
      One year             1473      166           11.27
      Two year             1695       48            2.83


In [4]:
# Q3: Which payment methods have highest churn?
query3 = """
SELECT 
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate_pct DESC;
"""

q3 = pd.read_sql_query(query3, conn)
print("Q3: Churn by Payment Method")
print("=" * 65)
print(q3.to_string(index=False))

Q3: Churn by Payment Method
            PaymentMethod  total_customers  churned  churn_rate_pct
         Electronic check             2365     1071           45.29
             Mailed check             1612      308           19.11
Bank transfer (automatic)             1544      258           16.71
  Credit card (automatic)             1522      232           15.24


In [6]:
# Q4: Does internet service type influence churn?
query4 = """
SELECT 
    InternetService,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY InternetService
ORDER BY churn_rate_pct DESC;
"""

q4 = pd.read_sql_query(query4, conn)
print("Q4: Churn by Internet Service")
print("=" * 55)
print(q4.to_string(index=False))                                 

Q4: Churn by Internet Service
InternetService  total_customers  churned  churn_rate_pct
    Fiber optic             3096     1297           41.89
            DSL             2421      459           18.96
             No             1526      113            7.40


In [7]:
# Q5: Does tenure affect churn?
query5 = """
SELECT 
    CASE 
        WHEN tenure <= 12 THEN '0-12 months'
        WHEN tenure <= 24 THEN '13-24 months'
        WHEN tenure <= 48 THEN '25-48 months'
        ELSE '49+ months'
    END AS tenure_band,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY tenure_band
ORDER BY churn_rate_pct DESC;
"""

q5 = pd.read_sql_query(query5, conn)
print("Q5: Churn by Tenure Band")
print("=" * 55)
print(q5.to_string(index=False))

Q5: Churn by Tenure Band
 tenure_band  total_customers  churned  churn_rate_pct
 0-12 months             2186     1037           47.44
13-24 months             1024      294           28.71
25-48 months             1594      325           20.39
  49+ months             2239      213            9.51


In [8]:
# Q6: Are high-paying customers more likely to churn?
query6 = """
SELECT 
    Churn,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
    ROUND(AVG(tenure), 1) AS avg_tenure_months,
    ROUND(AVG(TotalCharges), 2) AS avg_total_charges
FROM customers
GROUP BY Churn;
"""

q6 = pd.read_sql_query(query6, conn)
print("Q6: Charges & Tenure by Churn Status")
print("=" * 55)
print(q6.to_string(index=False))

Q6: Charges & Tenure by Churn Status
Churn  avg_monthly_charges  avg_tenure_months  avg_total_charges
   No                61.27               37.6            2549.91
  Yes                74.44               18.0            1531.80


In [9]:
# Q7: Do senior citizens churn more?
query7 = """
SELECT 
    CASE WHEN SeniorCitizen = 1 THEN 'Senior' ELSE 'Non-Senior' END AS customer_type,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY SeniorCitizen;
"""

q7 = pd.read_sql_query(query7, conn)
print("Q7: Churn by Senior Citizen Status")
print("=" * 50)
print(q7.to_string(index=False))

Q7: Churn by Senior Citizen Status
customer_type  total_customers  churned  churn_rate_pct
   Non-Senior             5901     1393           23.61
       Senior             1142      476           41.68


In [10]:
# Q8: Which customer segment is highest risk?
query8 = """
SELECT 
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Contract, InternetService, PaymentMethod
HAVING total_customers > 50
ORDER BY churn_rate_pct DESC
LIMIT 5;
"""

q8 = pd.read_sql_query(query8, conn)
print("Q8: Top 5 Highest Risk Customer Segments")
print("=" * 75)
print(q8.to_string(index=False))

Q8: Top 5 Highest Risk Customer Segments
      Contract InternetService             PaymentMethod  total_customers  churned  churn_rate_pct
Month-to-month     Fiber optic          Electronic check             1307      789           60.37
Month-to-month     Fiber optic              Mailed check              201      102           50.75
Month-to-month     Fiber optic Bank transfer (automatic)              327      149           45.57
Month-to-month     Fiber optic   Credit card (automatic)              293      122           41.64
Month-to-month             DSL          Electronic check              474      192           40.51


In [11]:
# Save all queries to a reusable SQL file for portfolio
sql_script = """
-- TeleComX Customer Churn Analysis
-- SQL Analysis Script
-- Analyst: Adedayo Adeyinka | June 2026

-- Q1: Overall Churn Rate
SELECT Churn, COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS percentage
FROM customers GROUP BY Churn ORDER BY Churn DESC;

-- Q2: Churn by Contract Type
SELECT Contract, COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers GROUP BY Contract ORDER BY churn_rate_pct DESC;

-- Q3: Churn by Payment Method
SELECT PaymentMethod, COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers GROUP BY PaymentMethod ORDER BY churn_rate_pct DESC;

-- Q4: Churn by Internet Service
SELECT InternetService, COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers GROUP BY InternetService ORDER BY churn_rate_pct DESC;

-- Q5: Churn by Tenure Band
SELECT CASE WHEN tenure <= 12 THEN '0-12 months'
    WHEN tenure <= 24 THEN '13-24 months'
    WHEN tenure <= 48 THEN '25-48 months'
    ELSE '49+ months' END AS tenure_band,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers GROUP BY tenure_band ORDER BY churn_rate_pct DESC;

-- Q6: Charges and Tenure by Churn Status
SELECT Churn, ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
    ROUND(AVG(tenure), 1) AS avg_tenure_months,
    ROUND(AVG(TotalCharges), 2) AS avg_total_charges
FROM customers GROUP BY Churn;

-- Q7: Senior Citizen Churn
SELECT CASE WHEN SeniorCitizen = 1 THEN 'Senior' ELSE 'Non-Senior' END AS customer_type,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers GROUP BY SeniorCitizen;

-- Q8: Top 5 Highest Risk Segments
SELECT Contract, InternetService, PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customers GROUP BY Contract, InternetService, PaymentMethod
HAVING total_customers > 50 ORDER BY churn_rate_pct DESC LIMIT 5;
"""

with open('../sql/churn_queries.sql', 'w') as f:
    f.write(sql_script)

print("SQL script saved to /sql/churn_queries.sql")

SQL script saved to /sql/churn_queries.sql
